# Document Question Answering System using RAG

This project builds a simple RAG system that answers questions from a custom resume PDF.

## Objectives

- Load a custom PDF document.
- Extract and process its text.
- Create vector embeddings.
- Store embeddings in FAISS.
- Retrieve relevant information.
- Generate answers using a language model.

## What is RAG?

RAG combines information retrieval with answer generation.

The system first finds relevant information from a document and then uses a language model to generate an answer based on that information.

## Step 1: Import Libraries

In [160]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_core.documents import Document
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from transformers import pipeline

print("All libraries imported successfully!")

All libraries imported successfully!


## Step 2: Load the Resume PDF

In [161]:
pdf_path = "resume.pdf"

loader = PyPDFLoader(pdf_path)

documents = loader.load()

print("PDF loaded successfully!")
print("Number of pages:", len(documents))

PDF loaded successfully!
Number of pages: 2


## Step 3: Extract Text

In [162]:
full_text = "\n".join(
    [doc.page_content for doc in documents]
)

print(full_text)

PRIYSHA 
 
priysha2911@gmail.com ⋄ (+91) 8604966944 
⋄ Linkedin ⋄ Github ⋄ LeetCode ⋄ Portfolio 
 
EDUCATION 
 
DIT University (Dehradun)                                                                                                       Aug 2023 - May 2027 
Bachelor of Technology in Computer Science and Engineering                                                                                       CGPA: 8.2/10 
CITY MONTESSORI SCHOOL (Lucknow)  
Sr. Secondary                                                                                                                             ISC, Science (PCM), 2022 – 95.6% 
Secondary                                                                                                                                                             ICSE, 2020 – 91.2% 
EXPERIENCE  / INTERNSHIPS  
 
TATA MOTORS                                                                                                                                           May 202

## Step 4: Split the Resume into Sections

In [163]:
section_names = [
    "EDUCATION",
    "EXPERIENCE / INTERNSHIPS",
    "PROJECTS",
    "OPEN SOURCE CONTRIBUTIONS",
    "CERTIFICATIONS",
    "DOMAIN / TECHNICAL SKILLS",
    "LEADERSHIP/EXTRACURRICULAR"
]

section_documents = []

for i, section in enumerate(section_names):

    start = full_text.find(section)

    if start == -1:
        continue

    end = len(full_text)

    for next_section in section_names[i + 1:]:

        next_start = full_text.find(
            next_section,
            start + len(section)
        )

        if next_start != -1:
            end = next_start
            break

    section_text = full_text[start:end].strip()

    section_documents.append(
        Document(
            page_content=section_text,
            metadata={"section": section}
        )
    )

print("Resume split into sections successfully!")
print("Number of sections:", len(section_documents))

Resume split into sections successfully!
Number of sections: 6


## Step 5: View the Resume Sections

In [164]:
for i, document in enumerate(section_documents):

    print("=" * 70)
    print("SECTION", i + 1)
    print("=" * 70)

    print(document.page_content)
    print()

SECTION 1
EDUCATION 
 
DIT University (Dehradun)                                                                                                       Aug 2023 - May 2027 
Bachelor of Technology in Computer Science and Engineering                                                                                       CGPA: 8.2/10 
CITY MONTESSORI SCHOOL (Lucknow)  
Sr. Secondary                                                                                                                             ISC, Science (PCM), 2022 – 95.6% 
Secondary                                                                                                                                                             ICSE, 2020 – 91.2% 
EXPERIENCE  / INTERNSHIPS  
 
TATA MOTORS                                                                                                                                           May 2026 - June 2026 
Full Stack  Developer  Intern                                             

## Step 6: Create Text Embeddings

In [165]:
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

print("Embedding model loaded successfully!")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 7611.23it/s]


Embedding model loaded successfully!


## Step 7: Create the FAISS Vector Database

In [166]:
vectorstore = FAISS.from_documents(
    section_documents,
    embeddings
)

print("FAISS vector database created successfully!")

FAISS vector database created successfully!


## Step 8: Create the Retriever

In [ ]:
retriever = vectorstore.as_retriever(
    search_kwargs={"k": 2}
)

print("Retriever created successfully!")

Retriever created successfully!


## Step 9: Test Information Retrieval

In [168]:
question = "What are Priysha's technical skills?"

results = retriever.invoke(question)

print("Question:", question)

print("\nRetrieved Information:")

for i, result in enumerate(results):

    print("\n" + "=" * 70)
    print("RESULT", i + 1)
    print("=" * 70)

    print(result.page_content)

Question: What are Priysha's technical skills?

Retrieved Information:

RESULT 1
DOMAIN / TECHNICAL SKILLS  
 
• Languages:  Java, Python, JavaScript, C, R   
• Frontend: React.js, HTML5, CSS3  
• Backend: Node.js, Express.js, REST API Development, API Integration, JSON   
• Databases: MongoDB, SQL, Database Normalization  
• ML/DL/GenAI: Regression, Classification, Clustering, Random Forest, XGBoost, ANN, CNN, RAG, LLMs, Prompt 
Engineering, Agentic AI    
• Tools & Version Control:  Git, GitHub, VS Code, Postman  
• Operating Systems:  Linux, Windows. 
• LeetCode: 200+ problems solved  · Contest Rating:  1480

RESULT 2
LEADERSHIP/EXTRACURRICULAR  
 
• Management Team, Sphurti:  Managed on-ground coordination across the event, logistics and hospitality for 
teams traveling from other colleges and states to participate in the college's inter -collegiate sports fest.  
• Google Developer Student Clubs (GDSC):  Completed a Generative AI -focused learning track, earning a Certificate 
of 

## Step 10: Create a Retrieval Function

In [175]:
def retrieve_information(question):
    
    results = retriever.invoke(question)
    
    context = "\n\n".join(
        [result.page_content for result in results]
    )
    
    return context

## Step 11: Load the Language Model

In [177]:
generator = pipeline(
    "text-generation",
    model="Qwen/Qwen2.5-0.5B-Instruct",
    max_new_tokens=80
)

print("Language model loaded successfully!")

Loading weights: 100%|██████████| 290/290 [00:00<00:00, 5273.04it/s]


Language model loaded successfully!


## Step 12: Create the RAG Prompt

In [176]:
def create_prompt(question, context):

    prompt = f"""You are a question answering system for a resume.

Answer the question using ONLY the information in the context.

Rules:
- Give a direct and concise answer.
- Do not add information that is not present in the context.
- Do not repeat the question.
- Do not give explanations about the context.
- Do not say "according to the resume".
- If the answer is not present in the context, say "The information is not available in the resume."

Context:
{context}

Question:
{question}

Answer:"""

    return prompt

## Step 13: Generate an Answer

In [179]:
def generate_answer(question):

    # Retrieve relevant information
    context = retrieve_information(question)

    # Create prompt
    prompt = create_prompt(question, context)

    # Generate answer
    response = generator(
        prompt,
        return_full_text=False
    )

    answer = response[0]["generated_text"].strip()

    return answer

## Step 14: Test the Complete RAG System

In [180]:
question = "What are Priysha's technical skills?"

answer = generate_answer(question)

print("=" * 70)
print("RAG DOCUMENT QUESTION ANSWERING SYSTEM")
print("=" * 70)

print("\nQuestion:")
print(question)

print("\nAnswer:")
print(answer)

[transformers] Both `max_new_tokens` (=80) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


RAG DOCUMENT QUESTION ANSWERING SYSTEM

Question:
What are Priysha's technical skills?

Answer:
Priyasha possesses a wide range of technical skills, including:

1. Java
2. Python
3. JavaScript
4. C
5. R
6. React.js
7. HTML5
8. CSS3
9. Node.js
10. Express.js
11. REST API Development
12. API Integration
13. JSON
14


## Step 15: Test Multiple Questions

In [181]:
questions = [
    "What are Priysha's technical skills?",
    "Where did Priysha intern?",
    "What projects has Priysha worked on?",
    "What programming languages does Priysha know?",
    "How many LeetCode problems has Priysha solved?",
    "What certifications does Priysha have?"
]

for question in questions:

    answer = generate_answer(question)

    print("=" * 70)
    print("QUESTION:")
    print(question)

    print("\nANSWER:")
    print(answer)

    print()

[transformers] Both `max_new_tokens` (=80) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=80) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


QUESTION:
What are Priysha's technical skills?

ANSWER:
Prysh's technical skills include languages like Java, Python, JavaScript, C, R; frontend technologies like React.js, HTML5, CSS3; backend technologies like Node.js, Express.js, REST API development, API integration, JSON; databases like MongoDB, SQL, database normalization; machine learning, deep learning, genai with regression, classification, clustering, random forest, xgboost



[transformers] Both `max_new_tokens` (=80) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


QUESTION:
Where did Priysha intern?

ANSWER:
TATA Motors
Explanation: The context provided indicates that TATA Motors was the employer of Priysha during her internship. However, the specific location where she interned is not mentioned in the given context. Therefore, the most accurate answer based solely on the provided information would be:

"Priysha interned at TATA Motors." 

This answer directly addresses the location where Priysha worked



[transformers] Both `max_new_tokens` (=80) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


QUESTION:
What projects has Priysha worked on?

ANSWER:
Priysha has worked on multiple projects. She has:

- Merged 8+ pull requests across 4 repositories
- Resolved issues and implemented features for these repositories. Additionally, she completed the GirlScript Summer of Code project, which included merging 6 pull requests to open-source repositories. This indicates her involvement in various tech-related projects and contributions to open-source communities. However, specific



[transformers] Both `max_new_tokens` (=80) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


QUESTION:
What programming languages does Priysha know?

ANSWER:
Priyasha knows the following programming languages:

1. Java
2. Python
3. JavaScript
4. C
5. R
6. Node.js
7. Express.js
8. REST API Development
9. API Integration
10. JSON
11. MongoDB
12. SQL
13. Database Normalization
14. Regression
15



[transformers] Both `max_new_tokens` (=80) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


QUESTION:
How many LeetCode problems has Priysha solved?

ANSWER:
Priyasha has solved 200 LeetCode problems on LeetCode. The information provided does not contain this specific number. To find out the exact number, please check the official LeetCode leaderboard or reach out to their support team for more details. The given text only provides basic skills and certifications related to programming languages and tools, but it lacks the specifics of solving actual LeetCode problems.

QUESTION:
What certifications does Priysha have?

ANSWER:
Priysha has certifications in Generative AI and Cloud Computing Foundations. Additionally, she holds a Certificate of Participation with Google Developer Student Clubs. The information provided in the context directly answers the question without adding any additional details or repeating anything. 

The information is not available in the resume. According to the context, there are no certifications mentioned for Priysha. The certification list include

## Conclusion

This project demonstrates a simple RAG system for answering questions from a custom PDF.

The system extracts information from the resume, creates embeddings, stores them in FAISS, retrieves relevant information, and generates answers using a language model.